# 01 — Extract the DOL LCA workbooks

**Inputs:** `data/lca/LCA_FY*.xlsx` (15 workbooks, ~1.8 GB, acquired manually in `00_pull`)
**Function:** stream each workbook and keep only the ~20 columns the analysis needs, reconciling three different column layouts
**Outputs:** `data/lca_slim/lca_*.csv` (15 files, ~600 MB)

Three things make this harder than a plain read:

1. The workbooks are 56–290 MB each, so they are streamed row by row in read-only mode
   rather than loaded into memory.
2. There are **three different column layouts**: 52 columns for FY2017–FY2018, 260 for
   FY2019 (where the wage fields repeat for every worksite), and 96 for FY2020 onward.
3. Column *names* differ too — `TOTAL_WORKER_POSITIONS` in some years,
   `TOTAL_WORKERS` in others.

The fix is an alias map: each canonical field lists every header it goes by, and the
extractor takes the first one present.

In [1]:
import sys
from pathlib import Path

# Locate the repository root by walking up until code/src is found, then put the
# code directory on the path. No absolute paths, so this runs from any checkout.
_here = Path.cwd().resolve()
_root = next(p for p in (_here, *_here.parents) if (p / "code" / "src").is_dir())
sys.path.insert(0, str(_root / "code"))

import csv

import openpyxl
import pandas as pd

from src import LCA_RAW, LCA_SLIM

## Functions

Defined up front, called below.

In [2]:
# Canonical field name -> the headers it goes by across the three layouts.
WANT = {
    "case_status":      ["CASE_STATUS"],
    "visa_class":       ["VISA_CLASS"],
    "decision_date":    ["DECISION_DATE"],
    "employer_name":    ["EMPLOYER_NAME"],
    "employer_state":   ["EMPLOYER_STATE"],
    "naics":            ["NAICS_CODE"],
    "soc_code":         ["SOC_CODE"],
    "job_title":        ["JOB_TITLE"],
    "full_time":        ["FULL_TIME_POSITION"],
    "n_workers":        ["TOTAL_WORKER_POSITIONS", "TOTAL_WORKERS"],
    "new_employment":   ["NEW_EMPLOYMENT"],
    "cont_employment":  ["CONTINUED_EMPLOYMENT"],
    "wage_from":        ["WAGE_RATE_OF_PAY_FROM", "WAGE_RATE_OF_PAY_FROM_1"],
    "wage_to":          ["WAGE_RATE_OF_PAY_TO", "WAGE_RATE_OF_PAY_TO_1"],
    "wage_unit":        ["WAGE_UNIT_OF_PAY", "WAGE_UNIT_OF_PAY_1"],
    "prevailing_wage":  ["PREVAILING_WAGE", "PREVAILING_WAGE_1"],
    "pw_unit":          ["PW_UNIT_OF_PAY", "PW_UNIT_OF_PAY_1"],
    "pw_level":         ["PW_WAGE_LEVEL", "PW_WAGE_LEVEL_1"],
    "h1b_dependent":    ["H_1B_DEPENDENT", "H1B_DEPENDENT", "H-1B_DEPENDENT"],
    "willful_violator": ["WILLFUL_VIOLATOR"],
    "agent_used":       ["AGENT_REPRESENTING_EMPLOYER"],
}
FIELDS = list(WANT)


def resolve_columns(header):
    """Map each canonical field to its column index in this workbook's header.

    Returns (index_map, missing_fields). A field absent from this layout maps to
    None and is written as empty rather than aborting the extract.
    """
    idx = {}
    for canon_name, aliases in WANT.items():
        idx[canon_name] = next((header.index(a) for a in aliases if a in header), None)
    missing = [k for k, v in idx.items() if v is None]
    return idx, missing


def extract(tag, overwrite=False):
    """Stream one workbook to a slim CSV.

    `tag` is a fiscal year ("2017") for the cumulative FY2017-FY2019 files, or a
    year-quarter ("2020_Q1") for FY2020 onward where DOL publishes quarterly.
    """
    src = LCA_RAW / f"LCA_FY{tag}.xlsx"
    dst = LCA_SLIM / f"lca_{tag}.csv"
    if dst.exists() and not overwrite:
        print(f"{tag}: already extracted, skipping")
        return dst
    if not src.exists():
        print(f"{tag}: SOURCE MISSING ({src.name}) -- see 00_pull")
        return None

    wb = openpyxl.load_workbook(src, read_only=True)
    ws = wb[wb.sheetnames[0]]
    rows = ws.iter_rows(values_only=True)
    header = [str(h).strip() if h is not None else "" for h in next(rows)]

    idx, missing = resolve_columns(header)
    print(f"{tag}: {len(header)} columns in source", end="")
    if missing:
        print(f"  WARNING missing {missing}", end="")
    print()

    n = 0
    with open(dst, "w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(FIELDS)
        for r in rows:
            if r is None or all(v is None for v in r):
                continue
            out = []
            for k in FIELDS:
                i = idx[k]
                v = r[i] if i is not None and i < len(r) else None
                out.append("" if v is None else str(v))
            writer.writerow(out)
            n += 1
    wb.close()
    print(f"{tag}: wrote {n:,} rows -> {dst.name}")
    return dst


def workbook_tags():
    """Fifteen tags: three annual files, then four quarters for each later year."""
    tags = ["2017", "2018", "2019"]
    for fy in (2020, 2021, 2022):
        tags += [f"{fy}_Q{q}" for q in (1, 2, 3, 4)]
    return tags

## Run the extraction

This is the slow step — roughly 20–40 minutes on a full cold run, since it parses
~4 million rows out of 1.8 GB of XML. Already-extracted files are skipped, so
re-running the notebook is cheap.

In [3]:
# Re-extract one workbook from scratch so the row count is visible in this run;
# the rest are skipped if their slim CSV already exists.
extract("2022_Q4", overwrite=True)
for tag in workbook_tags():
    extract(tag)


2022_Q4: 96 columns in source


2022_Q4: wrote 118,645 rows -> lca_2022_Q4.csv
2017: already extracted, skipping
2018: already extracted, skipping
2019: already extracted, skipping
2020_Q1: already extracted, skipping
2020_Q2: already extracted, skipping
2020_Q3: already extracted, skipping
2020_Q4: already extracted, skipping
2021_Q1: already extracted, skipping
2021_Q2: already extracted, skipping
2021_Q3: already extracted, skipping
2021_Q4: already extracted, skipping
2022_Q1: already extracted, skipping
2022_Q2: already extracted, skipping
2022_Q3: already extracted, skipping
2022_Q4: already extracted, skipping


### Diagnostic: what came out

Row counts per file. The FY2021 Q2/Q3 spike is real — it reflects the surge in filings
after the FY2021 cap season and the post-pandemic rebound, not a duplication bug.

In [4]:
rows = []
for tag in workbook_tags():
    p = LCA_SLIM / f"lca_{tag}.csv"
    if not p.exists():
        continue
    n = sum(1 for _ in open(p)) - 1
    rows.append({"file": p.name, "rows": n, "size_mb": round(p.stat().st_size / 1e6, 1)})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f"\nTOTAL extracted rows: {summary['rows'].sum():,}")
print(f"TOTAL size: {summary['size_mb'].sum() / 1000:.2f} GB")

           file   rows  size_mb
   lca_2017.csv 624650     93.0
   lca_2018.csv 654360     98.9
   lca_2019.csv 664616    101.0
lca_2020_Q1.csv 112017     18.9
lca_2020_Q2.csv 157173     23.6
lca_2020_Q3.csv 190749     28.4
lca_2020_Q4.csv 117395     17.7
lca_2021_Q1.csv  80622     13.1
lca_2021_Q2.csv 213481     37.4
lca_2021_Q3.csv 405626     70.8
lca_2021_Q4.csv 126576     22.2
lca_2022_Q1.csv 120306     18.7
lca_2022_Q2.csv 151603     23.4
lca_2022_Q3.csv 235530     36.0
lca_2022_Q4.csv 118645     18.6

TOTAL extracted rows: 3,973,349
TOTAL size: 0.62 GB


In [5]:
# Confirm the schema is identical across layouts -- this is the whole point of WANT.
first = pd.read_csv(LCA_SLIM / "lca_2017.csv", nrows=5, dtype=str)
last = pd.read_csv(LCA_SLIM / "lca_2022_Q4.csv", nrows=5, dtype=str)
print("FY2017 columns == FY2022Q4 columns:", list(first.columns) == list(last.columns))
print("columns:", list(first.columns))
first.head(3)

FY2017 columns == FY2022Q4 columns: True
columns: ['case_status', 'visa_class', 'decision_date', 'employer_name', 'employer_state', 'naics', 'soc_code', 'job_title', 'full_time', 'n_workers', 'new_employment', 'cont_employment', 'wage_from', 'wage_to', 'wage_unit', 'prevailing_wage', 'pw_unit', 'pw_level', 'h1b_dependent', 'willful_violator', 'agent_used']


,case_status,visa_class,decision_date,employer_name,employer_state,naics,soc_code,job_title,full_time,n_workers,...,cont_employment,wage_from,wage_to,wage_unit,prevailing_wage,pw_unit,pw_level,h1b_dependent,willful_violator,agent_used
0,CERTIFIED-WITHDRAWN,H-1B,2016-10-01 00:00:00,DISCOVER PRODUCTS INC.,IL,522210,15-1121,ASSOCIATE DATA INTEGRATION,Y,1,...,0,65811,67320,Year,59197,Year,Level I,N,N,Y
1,CERTIFIED-WITHDRAWN,H-1B,2016-10-01 00:00:00,DFS SERVICES LLC,IL,522210,15-2031,SENIOR ASSOCIATE,Y,1,...,0,53000,57200,Year,49800,Year,NaN,N,N,Y
2,CERTIFIED-WITHDRAWN,H-1B,2016-10-01 00:00:00,EASTBANC TECHNOLOGIES LLC,DC,541511,15-1131,.NET SOFTWARE PROGRAMMER,Y,2,...,0,77000,0,Year,76502,Year,Level II,Y,N,Y


**Next:** `02_merge.ipynb` cleans these rows, aggregates them to employer × fiscal year,
and joins them onto the USCIS panel.